# Spanish Learning From Preprocessed Data
This Colab-friendly notebook assumes you already have train.txt and valid.txt from preprocessing.

In [ ]:
!pip -q install transformers datasets accelerate sentencepiece evaluate

Mount your drive to connect the dataset (optional) ...

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# import preprocessed data
import shutil
from pathlib import Path

src = Path("/content/drive/MyDrive/cs273_spanish_project/processed")
dst = Path("/content/processed")

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)
print("Copied processed folder to:", dst)


In [ ]:
# import results (for continuing)
src_dir = Path("/content/drive/MyDrive/cs273_spanish_project/results")
dst_dir = Path("/content/report_outputs")
dst_dir.mkdir(parents=True, exist_ok=True)

for file_name in ["experiment_results.csv", "qualitative_predictions.csv"]:
    src = src_dir / file_name
    if src.exists():
        shutil.copy2(src, dst_dir / file_name)
        print("Copied:", file_name)


Or... Upload those two files when prompted, then run the remaining cells to train.

In [ ]:
from google.colab import files
from pathlib import Path
import os

Path("/content/processed").mkdir(parents=True, exist_ok=True)
uploaded = files.upload()

required = {"train.txt", "valid.txt"}
missing = required.difference(uploaded.keys())
if missing:
    raise ValueError(f"Missing required files: {sorted(missing)}")

for name in required:
    src = Path(name)
    dst = Path("/content/processed") / name
    if src.resolve() != dst.resolve():
        os.replace(src, dst)

print("Uploaded files to /content/processed")

In [ ]:
!wc -l /content/processed/train.txt /content/processed/valid.txt
!head -n 2 /content/processed/train.txt

   251422 /content/processed/train.txt
     5169 /content/processed/valid.txt
   256591 total
Acontecimientos . Nacimientos . Fallecimientos . Fulgencio de �cija, santo espa�ol. Erquinoaldo, mayordomo franco de palacio de Neustria. ;
Acontecimientos . Fin del Califato Perfecto. Los Omeyas en el poder. Califato de Damasco. Divisi�n entre sun�es y chi�es. Nacimientos . Fallecimientos .


## Sentence Piece tokenizing
With the train dataset we are going to tokenize over 2,000,000 sentences

In [ ]:
NEW_TOKENS = 2000  # try 2000 / 16000 / 30000 as ablations
SPM_VOCAB_SIZE = NEW_TOKENS + 8000  # overshoot so we can filter tokens already in BERT vocab

In [ ]:
import sentencepiece as spm

SPM_PREFIX = f"/content/spm_es_bpe_{SPM_VOCAB_SIZE}"
TRAIN_TXT = "/content/processed/train.txt"

# SentencePiece training (BPE)
spm.SentencePieceTrainer.Train(
    input=TRAIN_TXT,
    model_prefix=SPM_PREFIX,
    vocab_size=SPM_VOCAB_SIZE,
    model_type="bpe",
    character_coverage=0.9995,   # Spanish
    input_sentence_size=2_000_000,  # speed; can increase
    shuffle_input_sentence=True,
)

print("Saved:", SPM_PREFIX + ".model", SPM_PREFIX + ".vocab")

Saved: /content/spm_es_bpe_10000.model /content/spm_es_bpe_10000.vocab


In [ ]:
from transformers import AutoTokenizer, BertForMaskedLM
import sentencepiece as spm
import torch

base_model = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=True)
model = BertForMaskedLM.from_pretrained(base_model)

sp = spm.SentencePieceProcessor()
sp.load(SPM_PREFIX + ".model")

# SentencePiece piece strings
pieces = [sp.id_to_piece(i) for i in range(sp.get_piece_size())]

# Convert SP pieces into plain tokens.
# SentencePiece marks word starts with ▁. We drop that prefix so the added pieces are usable by BERT.
def normalize_sp_piece(p):
    if p.startswith("▁"):
        p = p[1:]
    return p

candidates = []
for p in pieces:
    p = normalize_sp_piece(p)
    if not p:
        continue
    if p in {"<unk>", "<s>", "</s>"}:
        continue
    if len(p) == 1 and not p.isalnum() and p not in {"ñ", "á", "é", "í", "ó", "ú", "ü"}:
        continue
    candidates.append(p)

bert_vocab = set(tokenizer.get_vocab().keys())
new_tokens = []
for t in candidates:
    if t in bert_vocab:
        continue
    if t in new_tokens:
        continue
    if len(t) > 20:
        continue
    new_tokens.append(t)
    if len(new_tokens) >= NEW_TOKENS:
        break

print("Planned new tokens:", len(new_tokens))
print("Sample:", new_tokens[:30])

added = tokenizer.add_tokens(new_tokens)
print("Actually added:", added)

model.resize_token_embeddings(len(tokenizer))

In [ ]:
with torch.no_grad():
    emb = model.get_input_embeddings().weight
    for tok in new_tokens:
        tok_id = tokenizer.convert_tokens_to_ids(tok)
        pieces_old = tokenizer.tokenize(tok)
        piece_ids = tokenizer.convert_tokens_to_ids(pieces_old)
        piece_ids = [i for i in piece_ids if i != tokenizer.unk_token_id]
        if len(piece_ids) > 0:
            emb[tok_id] = emb[piece_ids].mean(dim=0)

print("Initialized new embeddings.")

## Experiment Runner

The remaining cells are organized to run one experiment at a time from a 3x3 grid: token counts {500, 1000, 2000} crossed with article caps {full data, 100000, 50000}. Each run saves the model, evaluates it, and appends a row to a cumulative results table for the report.

In [ ]:
!pip -q install pandas matplotlib seaborn

In [ ]:
import math
import random
import statistics
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from transformers import AutoTokenizer, BertForMaskedLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, pipeline
from datasets import load_dataset

sns.set_theme(style="whitegrid")
REPORT_DIR = Path("/content/report_outputs")
REPORT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = REPORT_DIR / "experiment_results.csv"
QUALITATIVE_CSV = REPORT_DIR / "qualitative_predictions.csv"
print("Report outputs will be saved to:", REPORT_DIR)

In [ ]:
raw_ds = load_dataset(
    "text",
    data_files={
        "train": "/content/processed/train.txt",
        "validation": "/content/processed/valid.txt",
    },
)

TOTAL_TRAIN_ARTICLES = len(raw_ds["train"])
TOTAL_VALID_ARTICLES = len(raw_ds["validation"])
print({
    "train_articles": TOTAL_TRAIN_ARTICLES,
    "validation_articles": TOTAL_VALID_ARTICLES,
})

## Experiment Grid

These are the nine report models. You only need to change RUN_CONFIG in the next cell to train one of them. Re-run the same block with a different configuration to accumulate all nine rows in the results table.

In [ ]:
TOKEN_OPTIONS = [500, 1000, 2000]
ARTICLE_CAP_OPTIONS = [
    {"label": "full", "article_cap": TOTAL_TRAIN_ARTICLES},
    {"label": "100k", "article_cap": 100000},
    {"label": "50k", "article_cap": 50000},
]

experiment_grid = []
for token_count in TOKEN_OPTIONS:
    for article_spec in ARTICLE_CAP_OPTIONS:
        experiment_grid.append({
            "token_count": token_count,
            "article_label": article_spec["label"],
            "article_cap": article_spec["article_cap"],
            "approx_pct_of_train": round(article_spec["article_cap"] / TOTAL_TRAIN_ARTICLES, 4),
        })

grid_df = pd.DataFrame(experiment_grid)
grid_df

In [ ]:
RUN_CONFIG = {
    "token_count": 2000,
    "article_cap": 300000,
    "batch_size": 16,
    "learning_rate": 5e-5,
    "weight_decay": 0.01,
    "num_train_epochs": 1,
    "fixed_max_steps": 10000,
    "max_length": 128,
    "mlm_probability": 0.15,
    "warmup_ratio": 0.1,
    "eval_mask_examples": 40,
    "eval_text_samples": 500,
    "masked_accuracy_top_k": 5,
    "seed": 1337,
}
RUN_CONFIG

{'token_count': 2000,
 'article_cap': 300000,
 'batch_size': 16,
 'learning_rate': 5e-05,
 'weight_decay': 0.01,
 'num_train_epochs': 1,
 'fixed_max_steps': 10000,
 'max_length': 128,
 'mlm_probability': 0.15,
 'warmup_ratio': 0.1,
 'eval_mask_examples': 40,
 'eval_text_samples': 500,
 'masked_accuracy_top_k': 5,
 'seed': 1337}

In [ ]:
def make_experiment_name(config):
    article_cap = config["article_cap"]
    return f"bert_es_tok{config['token_count']}_art{article_cap}"

def count_parameters(model):
    return sum(param.numel() for param in model.parameters())

def subset_raw_dataset(dataset_split, cap):
    cap = min(cap, len(dataset_split))
    return dataset_split.select(range(cap))

def build_adapted_model_and_tokenizer(token_count):
    adapted_tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=True)
    adapted_model = BertForMaskedLM.from_pretrained(base_model)

    selected_tokens = new_tokens[:token_count]
    added = adapted_tokenizer.add_tokens(selected_tokens)
    adapted_model.resize_token_embeddings(len(adapted_tokenizer))

    with torch.no_grad():
        emb = adapted_model.get_input_embeddings().weight
        for tok in selected_tokens:
            tok_id = adapted_tokenizer.convert_tokens_to_ids(tok)
            pieces_old = adapted_tokenizer.tokenize(tok)
            piece_ids = adapted_tokenizer.convert_tokens_to_ids(pieces_old)
            piece_ids = [i for i in piece_ids if i != adapted_tokenizer.unk_token_id]
            if len(piece_ids) > 0:
                emb[tok_id] = emb[piece_ids].mean(dim=0)

    return adapted_model, adapted_tokenizer, selected_tokens, added

def tokenize_split(split_ds, tokenizer_obj, max_length):
    def _tok(batch):
        return tokenizer_obj(
            batch["text"],
            truncation=True,
            max_length=max_length,
            return_special_tokens_mask=True,
        )
    return split_ds.map(_tok, batched=True, remove_columns=["text"])

def tokenization_efficiency_for_tokenizer(tok, text_lines, n=500):
    rng = random.Random(0)
    sample = [text_lines[rng.randrange(len(text_lines))] for _ in range(min(n, len(text_lines)))]
    ratios = []
    for line in sample:
        words = line.strip().split()
        if not words:
            continue
        pieces = tok.tokenize(line)
        ratios.append(len(pieces) / max(1, len(words)))
    return statistics.mean(ratios), statistics.pstdev(ratios)

def build_masked_examples(text_lines, tokenizer_obj, max_examples=40, max_length=128):
    masked = []
    strip_chars = ".,;:!?¿¡\"'()[]{}"
    for line in text_lines:
        words = line.split()
        if len(words) < 8:
            continue
        idx = len(words) // 2
        target = words[idx].strip(strip_chars)
        if len(target) < 4:
            continue
        masked_words = words.copy()
        masked_words[idx] = tokenizer_obj.mask_token
        masked_text = " ".join(masked_words)
        if len(tokenizer_obj.tokenize(masked_text)) >= max_length:
            continue
        masked.append((masked_text, target.lower(), line))
        if len(masked) >= max_examples:
            break
    return masked

def evaluate_masked_accuracy(model_obj, tokenizer_obj, masked_examples, top_k=5):
    fill_mask = pipeline(
        "fill-mask",
        model=model_obj,
        tokenizer=tokenizer_obj,
        device=0 if torch.cuda.is_available() else -1,
    )
    hits = 0
    rows = []
    for masked_text, target, original in masked_examples:
        preds = fill_mask(masked_text, top_k=top_k)
        pred_tokens = [p["token_str"].strip().lower() for p in preds]
        hit = target in pred_tokens
        hits += int(hit)
        rows.append({
            "masked_text": masked_text,
            "target": target,
            "predictions": ", ".join(pred_tokens),
            "hit@5": hit,
            "original_text": original,
        })
    return hits / max(1, len(masked_examples)), rows

def upsert_result_row(result_row):
    row_df = pd.DataFrame([result_row])
    if RESULTS_CSV.exists():
        existing = pd.read_csv(RESULTS_CSV)
        existing = existing[existing["experiment_name"] != result_row["experiment_name"]]
        combined = pd.concat([existing, row_df], ignore_index=True)
    else:
        combined = row_df
    combined = combined.sort_values(["token_count", "article_cap"]).reset_index(drop=True)
    combined.to_csv(RESULTS_CSV, index=False)
    return combined

## Run One Experiment

This is the main block to run. Change only RUN_CONFIG, then execute this cell. It will train one model, evaluate overall loss, perplexity, token ratio, and masked accuracy, save the model, and append the metrics to experiment_results.csv.

In [ ]:
experiment_name = make_experiment_name(RUN_CONFIG)
model_dir = Path("/content/experiments") / experiment_name
checkpoint_dir = model_dir / "checkpoints"
final_dir = model_dir / "final"
model_dir.mkdir(parents=True, exist_ok=True)

train_cap = min(RUN_CONFIG["article_cap"], TOTAL_TRAIN_ARTICLES)
valid_cap = max(500, round(TOTAL_VALID_ARTICLES * (train_cap / TOTAL_TRAIN_ARTICLES)))
valid_cap = min(valid_cap, TOTAL_VALID_ARTICLES)

train_subset = subset_raw_dataset(raw_ds["train"], train_cap)
valid_subset = subset_raw_dataset(raw_ds["validation"], valid_cap)
valid_lines = [row["text"].strip() for row in valid_subset if row["text"].strip()]

model, tokenizer, selected_tokens, added_tokens = build_adapted_model_and_tokenizer(RUN_CONFIG["token_count"])
train_tokenized = tokenize_split(train_subset, tokenizer, RUN_CONFIG["max_length"])
valid_tokenized = tokenize_split(valid_subset, tokenizer, RUN_CONFIG["max_length"])

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=RUN_CONFIG["mlm_probability"],
)

fixed_max_steps = RUN_CONFIG["fixed_max_steps"]
warmup_steps = max(1, int(fixed_max_steps * RUN_CONFIG["warmup_ratio"]))

args = TrainingArguments(
    output_dir=str(checkpoint_dir),
    per_device_train_batch_size=RUN_CONFIG["batch_size"],
    per_device_eval_batch_size=RUN_CONFIG["batch_size"],
    learning_rate=RUN_CONFIG["learning_rate"],
    weight_decay=RUN_CONFIG["weight_decay"],
    num_train_epochs=RUN_CONFIG["num_train_epochs"],
    max_steps=fixed_max_steps,
    warmup_steps=warmup_steps,
    seed=RUN_CONFIG["seed"],
    data_seed=RUN_CONFIG["seed"],
    eval_strategy="steps",
    eval_steps=max(100, fixed_max_steps // 5),
    save_strategy="steps",
    save_steps=max(100, fixed_max_steps // 5),
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    data_collator=collator,
)

train_output = trainer.train()
eval_metrics = trainer.evaluate()
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

trained_model = trainer.model
trained_tokenizer = tokenizer

avg_ratio, std_ratio = tokenization_efficiency_for_tokenizer(
    trained_tokenizer,
    valid_lines,
    n=RUN_CONFIG["eval_text_samples"],
)
masked_examples = build_masked_examples(
    valid_lines,
    trained_tokenizer,
    max_examples=RUN_CONFIG["eval_mask_examples"],
    max_length=RUN_CONFIG["max_length"],
)
mask_accuracy, qualitative_rows = evaluate_masked_accuracy(
    trained_model,
    trained_tokenizer,
    masked_examples,
    top_k=RUN_CONFIG["masked_accuracy_top_k"],
)

eval_loss = float(eval_metrics["eval_loss"])
perplexity = math.exp(eval_loss)
result_row = {
    "experiment_name": experiment_name,
    "token_count": RUN_CONFIG["token_count"],
    "article_cap": train_cap,
    "approx_pct_of_train": round(train_cap / TOTAL_TRAIN_ARTICLES, 4),
    "train_articles": len(train_subset),
    "valid_articles": len(valid_subset),
    "added_tokens": added_tokens,
    "parameters_millions": round(count_parameters(trained_model) / 1_000_000, 2),
    "train_runtime_sec": round(train_output.metrics.get("train_runtime", 0.0), 2),
    "train_samples_per_sec": round(train_output.metrics.get("train_samples_per_second", 0.0), 4),
    "eval_loss": round(eval_loss, 4),
    "perplexity": round(perplexity, 4),
    "token_ratio": round(avg_ratio, 4),
    "token_ratio_std": round(std_ratio, 4),
    "masked_accuracy": round(mask_accuracy, 4),
    "model_dir": str(final_dir),
}

results_df = upsert_result_row(result_row)
qualitative_df = pd.DataFrame(qualitative_rows)
qualitative_df.insert(0, "experiment_name", experiment_name)
if QUALITATIVE_CSV.exists():
    existing_qual = pd.read_csv(QUALITATIVE_CSV)
    existing_qual = existing_qual[existing_qual["experiment_name"] != experiment_name]
    qualitative_df = pd.concat([existing_qual, qualitative_df], ignore_index=True)
qualitative_df.to_csv(QUALITATIVE_CSV, index=False)

pd.DataFrame([result_row])

### Saving files to my drive (Optional)
Just the final models

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

DRIVE_EXPERIMENTS_DIR = Path("/content/drive/MyDrive/cs273_spanish_project/experiments")
DRIVE_EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
experiment_name = make_experiment_name(RUN_CONFIG)

local_final_dir = Path("/content/experiments") / experiment_name / "final"
drive_final_dir = DRIVE_EXPERIMENTS_DIR / experiment_name / "final"

drive_final_dir.parent.mkdir(parents=True, exist_ok=True)

if drive_final_dir.exists():
    shutil.rmtree(drive_final_dir)

shutil.copytree(local_final_dir, drive_final_dir)
print("Saved final model only to:", drive_final_dir)

In [ ]:
models_to_save = [
    "bert_es_tok500_art50000",
    "bert_es_tok500_art100000",
    "bert_es_tok500_art300000",
    "bert_es_tok1000_art50000",
    "bert_es_tok1000_art100000",
    "bert_es_tok1000_art300000",
    "bert_es_tok2000_art50000",
    "bert_es_tok2000_art100000",
    "bert_es_tok2000_art300000",
]

for model_name in models_to_save:
    src = Path("/content/experiments") / model_name / "final"
    dst = Path("/content/drive/MyDrive/cs273_spanish_project/experiments") / model_name / "final"

    dst.parent.mkdir(parents=True, exist_ok=True)

    if not src.exists():
        print("Missing:", src)
        continue

    if dst.exists():
        shutil.rmtree(dst)

    shutil.copytree(src, dst)
    print("Saved final model:", dst)


In [ ]:
DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/cs273_spanish_project/results")
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for file_name in ["experiment_results.csv", "qualitative_predictions.csv"]:
    src = Path("/content/report_outputs") / file_name
    if src.exists():
        shutil.copy2(src, DRIVE_RESULTS_DIR / file_name)
        print("Saved:", DRIVE_RESULTS_DIR / file_name)



## Test The Most Recently Trained Model

After running one experiment, use this cell to inspect tokenization and masked predictions on real text.

In [ ]:
from transformers import pipeline

model_path = "/content/experiments/bert_es_tok500_art300000/"

saved_tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
saved_model = BertForMaskedLM.from_pretrained(model_path)
saved_model = saved_model.to("cuda" if torch.cuda.is_available() else "cpu")

test_text = "Me gusta aprender [MASK] porque puedo hablar con mas personas."
fill_mask = pipeline(
    "fill-mask",
    model=trained_model_path,
    tokenizer=trained_tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

print("Input text:")
print(test_text)
print()
print("Tokenizer output:")
print(trained_tokenizer.tokenize(test_text))
print()
print("Top mask predictions:")
for pred in fill_mask(test_text, top_k=5):
    print(f"{pred['token_str']!r}: score={pred['score']:.4f} | {pred['sequence']}")

## Results Table

This cell reloads the cumulative results CSV so you can monitor progress as you complete the 9 runs.

In [ ]:
if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV).sort_values(["token_count", "article_cap"]).reset_index(drop=True)
    display(results_df)
else:
    print("No experiment results saved yet.")

In [ ]:
if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV)
    loss_table = results_df.pivot(index="token_count", columns="article_cap", values="eval_loss")
    ppl_table = results_df.pivot(index="token_count", columns="article_cap", values="perplexity")
    ratio_table = results_df.pivot(index="token_count", columns="article_cap", values="token_ratio")
    acc_table = results_df.pivot(index="token_count", columns="article_cap", values="masked_accuracy")

    print("Loss table")
    display(loss_table)
    print("Perplexity table")
    display(ppl_table)
    print("Token ratio table")
    display(ratio_table)
    print("Masked accuracy table")
    display(acc_table)
else:
    print("Run at least one experiment first.")

In [ ]:
if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    sns.heatmap(results_df.pivot(index="token_count", columns="article_cap", values="eval_loss"), annot=True, fmt=".4f", cmap="mako", ax=axes[0, 0])
    axes[0, 0].set_title("Eval Loss")

    sns.heatmap(results_df.pivot(index="token_count", columns="article_cap", values="perplexity"), annot=True, fmt=".2f", cmap="crest", ax=axes[0, 1])
    axes[0, 1].set_title("Perplexity")

    sns.heatmap(results_df.pivot(index="token_count", columns="article_cap", values="token_ratio"), annot=True, fmt=".4f", cmap="flare", ax=axes[1, 0])
    axes[1, 0].set_title("Token Ratio")

    sns.heatmap(results_df.pivot(index="token_count", columns="article_cap", values="masked_accuracy"), annot=True, fmt=".4f", cmap="viridis", ax=axes[1, 1])
    axes[1, 1].set_title("Masked Accuracy")

    for ax in axes.flat:
        ax.set_xlabel("Training articles")
        ax.set_ylabel("Added tokens")

    plt.tight_layout()
    figure_path = REPORT_DIR / "experiment_heatmaps.png"
    plt.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved figure to:", figure_path)
else:
    print("Run at least one experiment first.")